In [143]:
import pandas as pd

df = pd.read_csv("/vol/corpora/CommonVoice/cv-corpus-19.0-2024-09-13/fr/train.tsv", sep="\t")

# Keep only what we need
df = df[["path", "sentence"]]

# Remove missing or empty sentences
df = df.dropna(subset=["sentence"])
df = df[df["sentence"].str.strip() != ""]

print("Remaining rows:", len(df))


/tmp/ipykernel_2335502/1172488694.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/vol/corpora/CommonVoice/cv-corpus-19.0-2024-09-13/fr/train.tsv", sep="\t")


Remaining rows: 570806


In [144]:
df_small = df

In [145]:
from phonemizer import phonemize
from phonemizer.separator import Separator

separator = Separator(phone=" ", word=" | ")
def normalize_phoneme(ph):
    return ph.replace("ː", "")
def phonemize_texts(texts):
    return phonemize(
        texts,
        language="fr-fr",
        backend="espeak",
        strip=True,
        preserve_punctuation=False,
        with_stress=False,
        separator=separator,
        njobs=4  # use CPU cores
    )

sentences = df_small["sentence"].tolist()

phoneme_list = phonemize_texts(sentences)
phoneme_list1 =[normalize_phoneme(ph) for ph in phoneme_list]
    
df_small["phonemes"] = phoneme_list1

print(df_small.head())


                           path  \
0  common_voice_fr_22773552.mp3   
1  common_voice_fr_22773553.mp3   
2  common_voice_fr_22773554.mp3   
3  common_voice_fr_22773555.mp3   
4  common_voice_fr_22773556.mp3   

                                            sentence  \
0     Il mourut à Arras des suites de ses blessures.   
1  Il dispose de différentes options d'harmonie q...   
2  Il se produit autant à la radio qu'à la télévi...   
3       De nouveaux puits et fontaines sont creusés.   
4  Le film comporte deux facettes qui opposent la...   

                                            phonemes  
0  i l | m u ʁ y | a | a ʁ a | d e | s y i t | d ...  
1  i l | d i s p ɔ z | d ə | d i f e ʁ ɑ̃ t z | ɔ...  
2  i l | s ə | p ʁ o d y i | o t ɑ̃ | a | l a | ʁ...  
3  d ə | n u v o | p y i z | e | f ɔ̃ t ɛ n | s ɔ...  
4  l ə | f i l m | k ɔ̃ p ɔ ʁ t | d ø | f a s ɛ t...  


In [146]:
all_phonemes = set()

for row in df_small["phonemes"]:
    tokens = row.split()
    for t in tokens:
        all_phonemes.add(t)

print("Number of unique tokens:", len(all_phonemes))
print(sorted(all_phonemes))


Number of unique tokens: 73
['(el)', '(en)', '(fr)', '(ka)', '1', 'a', 'aɪ', 'aɪə', 'aʊ', 'b', 'c', 'd', 'dz', 'dʒ', 'e', 'eə', 'eɪ', 'f', 'h', 'i', 'iə', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'tʃ', 'u', 'v', 'w', 'x', 'y', 'z', '|', 'ç', 'ð', 'ø', 'ŋ', 'œ', 'œ̃', 'ɐ', 'ɑ', 'ɑ̃', 'ɒ', 'ɔ', 'ɔɪ', 'ɔ̃', 'ə', 'əl', 'əʊ', 'ɛ', 'ɛ̃', 'ɜ', 'ɡ', 'ɡʲ', 'ɣ', 'ɪ', 'ɬ', 'ɲ', 'ɹ', 'ʁ', 'ʃ', 'ʊ', 'ʊə', 'ʌ', 'ʒ', 'ʔ', 'θ']


filter dataset:

Remove sentences containing unwanted phonemes

Keep only rows with clean French inventory

In [147]:
allowed_phonemes = [
  "|","a", "b", "c", "d",  "e", "f", "i", "j", "k", "l", "m", "n",
    "o", "p", "s", "t",  "u", "v", "w", "y", "z", "ø", "ŋ", "œ",
    "ɑ", "ɑ̃", "ɔ", "ɔ̃", "ə", "ɛ", "ɛ̃", "ɟ", "ɡ", "ɥ", "ɲ", "ʁ", "ʃ", "ʎ", "ʒ",'œ̃'
]
len(allowed_phonemes)

41

In [148]:
def is_clean(phoneme_string):
    tokens = phoneme_string.split()
    return all(t in allowed_phonemes for t in tokens)

df_clean = df_small[df_small["phonemes"].apply(is_clean)]

print("Before:", len(df_small))
print("After cleaning:", len(df_clean))


Before: 570806
After cleaning: 507972


In [149]:
df =df_clean
df["phonemes"] = df["phonemes"].str.replace("|", "", regex=False)
df["phonemes"] = df["phonemes"].str.replace("  ", " ", regex=False)
df["phonemes"] = df["phonemes"].str.strip()
df.head()

/tmp/ipykernel_2335502/1650946763.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["phonemes"] = df["phonemes"].str.replace("|", "", regex=False)
/tmp/ipykernel_2335502/1650946763.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["phonemes"] = df["phonemes"].str.replace("  ", " ", regex=False)
/tmp/ipykernel_2335502/1650946763.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

,path,sentence,phonemes
0,common_voice_fr_22773552.mp3,Il mourut à Arras des suites de ses blessures.,i l m u ʁ y a a ʁ a d e s y i t d ə s e b l ɛ ...
2,common_voice_fr_22773554.mp3,Il se produit autant à la radio qu'à la télévi...,i l s ə p ʁ o d y i o t ɑ̃ a l a ʁ a d j o k a...
3,common_voice_fr_22773555.mp3,De nouveaux puits et fontaines sont creusés.,d ə n u v o p y i z e f ɔ̃ t ɛ n s ɔ̃ k ʁ ø z e
4,common_voice_fr_22773556.mp3,Le film comporte deux facettes qui opposent la...,l ə f i l m k ɔ̃ p ɔ ʁ t d ø f a s ɛ t k i ɔ p...
5,common_voice_fr_18047721.mp3,Cette bruyante et éloquente reconnaissance d'H...,s ɛ t b ʁ y i j ɑ̃ t e e l o k ɑ̃ t ʁ ə k ɔ n ...


In [150]:
import unicodedata

for idx, phoneme_string in df["phonemes"].items():
    tokens = phoneme_string.split()

    normalized_tokens = []
    for token in tokens:
        if "\u0303" in token:
            token = unicodedata.normalize("NFC", token)
        normalized_tokens.append(token)

    df.at[idx, "phonemes"] = " ".join(normalized_tokens)

In [151]:
import unicodedata

df["phonemes"] = df["phonemes"].apply(
    lambda x: " ".join(
        unicodedata.normalize("NFC", token)
        for token in x.split()
    )
)

/tmp/ipykernel_2335502/3142256242.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["phonemes"] = df["phonemes"].apply(


In [152]:
df.to_csv("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/commonvoice_fr_wavlm_train.csv", index=False)


In [142]:
all_phonemes = set()

for seq in df["phonemes"]:
    all_phonemes.update(seq.split())

print(sorted(all_phonemes))

['a', 'b', 'd', 'e', 'f', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 's', 't', 'u', 'v', 'w', 'y', 'z', 'ø', 'ŋ', 'œ', 'œ̃', 'ɑ̃', 'ɔ', 'ɔ̃', 'ə', 'ɛ', 'ɛ̃', 'ɡ', 'ɲ', 'ʁ', 'ʃ', 'ʒ']


Vocab dictionary

In [128]:
import json

vocab_list = sorted(list(all_phonemes))

# Add special tokens
vocab_list.append("[UNK]")
vocab_list.append("[PAD]")

vocab_dict = {v: k for k, v in enumerate(vocab_list)}

print("Final vocab size:", len(vocab_dict))

with open("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/vocab_CV.json", "w") as f:
    json.dump(vocab_dict, f, ensure_ascii=False, indent=2)


Final vocab size: 37


In [130]:
vocab_dict

{'a': 0,
 'b': 1,
 'd': 2,
 'e': 3,
 'f': 4,
 'i': 5,
 'j': 6,
 'k': 7,
 'l': 8,
 'm': 9,
 'n': 10,
 'o': 11,
 'p': 12,
 's': 13,
 't': 14,
 'u': 15,
 'v': 16,
 'w': 17,
 'y': 18,
 'z': 19,
 'ø': 20,
 'ŋ': 21,
 'œ': 22,
 'œ̃': 23,
 'ɑ̃': 24,
 'ɔ': 25,
 'ɔ̃': 26,
 'ə': 27,
 'ɛ': 28,
 'ɛ̃': 29,
 'ɡ': 30,
 'ɲ': 31,
 'ʁ': 32,
 'ʃ': 33,
 'ʒ': 34,
 '[UNK]': 35,
 '[PAD]': 36}

In [57]:
import json
import unicodedata
import re

def analyze_token(token):
    print(f"\nToken: {repr(token)}")
    for i, ch in enumerate(token):
        print(f"  {i}: {repr(ch)} → {unicodedata.name(ch)}")
        token = repr(unicodedata.normalize("NFC", token))

def contains_space_before_combining(token):
    return bool(re.search(r"\s[\u0300-\u036f]", token))

# Load vocab
with open("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/vocab_CV.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)

# If vocab is dict like {"token": id}
tokens = vocab.keys()

# Check all tokens
for token in tokens:
    # check if token contains combining tilde
    if "\u0303" in token:
        analyze_token(token)
        if contains_space_before_combining(token):
            print("⚠️  SPACE before combining mark detected!")



Token: 'ɑ̃'
  0: 'ɑ' → LATIN SMALL LETTER ALPHA
  1: '̃' → COMBINING TILDE

Token: 'ɔ̃'
  0: 'ɔ' → LATIN SMALL LETTER OPEN O
  1: '̃' → COMBINING TILDE

Token: 'ɛ̃'
  0: 'ɛ' → LATIN SMALL LETTER OPEN E
  1: '̃' → COMBINING TILDE


In [15]:
df_clean["id_speaker"]=[i.split("_")[-1].split(".")[0] for i in df_clean["path"]]

/tmp/ipykernel_2331810/1981353803.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["id_speaker"]=[i.split("_")[-1].split(".")[0] for i in df_clean["path"]]


In [16]:
df =df_clean
df["phonemes"] = df["phonemes"].str.replace("|", "", regex=False)
df["phonemes"] = df["phonemes"].str.replace("  ", " ", regex=False)
df["phonemes"] = df["phonemes"].str.strip()


/tmp/ipykernel_2331810/1601490823.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["phonemes"] = df["phonemes"].str.replace("|", "", regex=False)
/tmp/ipykernel_2331810/1601490823.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["phonemes"] = df["phonemes"].str.replace("  ", " ", regex=False)
/tmp/ipykernel_2331810/1601490823.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

In [18]:
df_clean.head

<bound method NDFrame.head of                                 path  \
0       common_voice_fr_22773552.mp3   
2       common_voice_fr_22773554.mp3   
3       common_voice_fr_22773555.mp3   
4       common_voice_fr_22773556.mp3   
5       common_voice_fr_18047721.mp3   
...                              ...   
570801  common_voice_fr_38156775.mp3   
570802  common_voice_fr_38156776.mp3   
570803  common_voice_fr_38156781.mp3   
570804  common_voice_fr_38156786.mp3   
570805  common_voice_fr_38156787.mp3   

                                                 sentence  \
0          Il mourut à Arras des suites de ses blessures.   
2       Il se produit autant à la radio qu'à la télévi...   
3            De nouveaux puits et fontaines sont creusés.   
4       Le film comporte deux facettes qui opposent la...   
5       Cette bruyante et éloquente reconnaissance d'H...   
...                                                   ...   
570801            Il sort en début de match sur blessure.   
5

In [40]:
import os
import subprocess
from pathlib import Path
from tqdm import tqdm

output_root = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/cv_mfa_train"

for _, row in tqdm(df_clean.iterrows(), total=len(df_clean)):

    audio_path = "/vol/corpora/CommonVoice/cv-corpus-19.0-2024-09-13/fr/clips/"+row["path"]        # path to mp3 or wav
    phoneme_seq = row["phonemes"]         # "ʒ a v ɛ l a ʃ ..."
    speaker_id = str(row["id_speaker"])

    # Create speaker folder
    speaker_dir = os.path.join(output_root, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)

    # Define output wav path
    filename_stem = Path(audio_path).stem
    wav_path = os.path.join(speaker_dir, filename_stem + ".wav")

    # Convert to 16kHz mono wav
    subprocess.run([
        "ffmpeg", "-y",
        "-i", audio_path,
        "-ar", "16000",
        "-ac", "1",
        wav_path
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Write phoneme transcript
    lab_path = wav_path.replace(".wav", ".lab")
    with open(lab_path, "w", encoding="utf8") as f:
        f.write(phoneme_seq.strip())


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7546/7546 [06:27<00:00, 19.48it/s]


CTC tokenizer

In [20]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer(
    "vocab.json",
    pad_token="[PAD]",
    word_delimiter_token=""
)

print("Tokenizer vocab size:", len(tokenizer))


Tokenizer vocab size: 40


Feature extractor

In [21]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)


In [22]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer
)


In [23]:
test = df_clean.iloc[0]["phonemes"]

print("Original:", test)

encoded = processor.tokenizer(test).input_ids
print("Encoded:", encoded)

decoded = processor.tokenizer.decode(encoded, group_tokens=False)
print("Decoded:", decoded)


Original: i l m u ʁ y a a ʁ a d e s y i t d ə s e b l ɛ s y ʁ
Encoded: [5, 8, 9, 15, 32, 18, 0, 0, 32, 0, 2, 3, 13, 18, 5, 14, 2, 27, 13, 3, 1, 8, 28, 13, 18, 32]
Decoded: ilmuʁyaaʁadesyitdəseblɛsyʁ


In [25]:
#Load Model With Correct Vocab Size
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-FR-7K-large",
    vocab_size=len(tokenizer),   # should be 40
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at /vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-FR-7K-large and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [26]:
#Freeze Feature Encoder (
model.freeze_feature_encoder()


In [27]:
print(model.lm_head.out_features)

40


In [36]:
import os
import torch
import librosa
import torchaudio
from torch.utils.data import Dataset

class CommonVoicePhonemeDataset(Dataset):
    def __init__(self, dataframe, audio_dir, processor):
        self.df = dataframe.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load audio
        audio_path = os.path.join(self.audio_dir, row["path"])

        waveform, sr = librosa.load(audio_path, sr=16000)
        waveform = torch.tensor(waveform)

        input_values = self.processor(waveform,sampling_rate=16000,return_tensors="pt").input_values.squeeze()

        # Process labels
        with self.processor.as_target_processor():
            labels = self.processor(row["phonemes"]).input_ids

        return {
            "input_values": input_values,
            "labels": torch.tensor(labels, dtype=torch.long)
        }


In [37]:
audio_directory = "/vol/corpora/CommonVoice/cv-corpus-19.0-2024-09-13/fr/clips"

train_dataset = CommonVoicePhonemeDataset(
    df_clean,
    audio_directory,
    processor
)


In [39]:
sample = train_dataset[0]

print("Input shape:", sample["input_values"].shape)
print("Labels:", sample["labels"])


Input shape: torch.Size([124992])
Labels: tensor([ 2, 20, 28, 20, 12, 20, 18, 20,  5, 20, 20, 20, 13, 20,  0, 20, 20, 20,
         7, 20, 33, 20,  3, 20,  0, 20, 13, 20,  6, 27, 20, 20,  8, 20, 28, 20,
        20, 20, 33, 20,  3, 20, 19, 20, 11, 20, 20, 20,  7, 20, 26, 20, 10, 20,
        29, 20, 14, 20, 20, 20, 18, 20, 10, 20, 20, 20,  4, 20, 33, 20,  3, 20,
         7, 25, 14, 20,  0, 20, 13, 20,  6, 27, 20, 20,  2, 20, 28, 20, 20, 20,
        12, 20,  8, 20, 18, 20, 19, 20, 20, 25, 20, 20, 12, 20,  8, 20, 18, 20,
        19, 20, 20, 30, 12, 20, 26, 20, 33, 20, 14, 25, 14])


In [40]:
from dataclasses import dataclass
from typing import List, Dict, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # Pad audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        # Pad labels
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # Replace padding with -100 for CTC
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels

        return batch


In [41]:
data_collator = DataCollatorCTCWithPadding(processor=processor)
batch = data_collator([train_dataset[0], train_dataset[1]])

print(batch["input_values"].shape)
print(batch["labels"].shape)


torch.Size([2, 124992])
torch.Size([2, 121])


In [43]:
import torch
print(torch.cuda.get_device_name(0))
print("Memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


NVIDIA A100-PCIE-40GB
Memory (GB): 42.505273344


In [55]:
training_args = TrainingArguments(
    output_dir="./wav2vec2-fr-phoneme",

    per_device_train_batch_size=4,   # reduced from 16
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,   # keeps effective batch = 16

    evaluation_strategy="steps",
    save_strategy="steps",

    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,

    num_train_epochs=15,

    learning_rate=1e-4,
    warmup_steps=1000,

    fp16=True,

    save_total_limit=3,
    report_to="none",
)


/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [56]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_clean,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))


Train size: 39015
Validation size: 4336


In [57]:
train_dataset = CommonVoicePhonemeDataset(
    train_df,
    audio_directory,
    processor
)

val_dataset = CommonVoicePhonemeDataset(
    val_df,
    audio_directory,
    processor
)


In [58]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")  # we'll use it for PER

def compute_metrics(pred):
    logits = pred.predictions
    pred_ids = np.argmax(logits, axis=-1)

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids)

    # Decode labels
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    # Compute PER
    per = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"per": per}


In [59]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)


/tmp/ipykernel_1253832/3829613150.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [60]:
model.to("cuda")
trainer.train()


/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/transformers/models/wav2vec2/processing_wav2vec2.py:174: UserWarning: `as_target_processor` is deprecated and will be removed in v5 of Transformers. You can process your labels by using the argument `text` of the regular `__call__` method (either in the same call as your audio inputs, or in a separate call.
  warnings.warn(
/home/imbenamor/miniconda3/envs/venv-test/lib/python3.8/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss


KeyboardInterrupt: 